In [0]:
landing_folder = "abfss://sales@trendytechstracc.dfs.core.windows.net/landing"
staging_folder = "abfss://sales@trendytechstracc.dfs.core.windows.net/staging"
discarded_folder = "abfss://sales@trendytechstracc.dfs.core.windows.net/discarded"

In [0]:
ordersDf = spark.read.csv(f'{landing_folder}/orders.csv', inferSchema=True, header=True)

In [0]:
display(ordersDf)

### 1 st Condition : move the file to discarded folder if it has duplicate order_id's

In [0]:
errorFlag = False

ordersCount = ordersDf.count()
print(ordersCount)

distinctOrdersCount = ordersDf.select('order_id').distinct().count()
print(distinctOrdersCount)

if ordersCount != distinctOrdersCount:
    errorFlag = True

if errorFlag:
    dbutils.fs.mv(f'{landing_folder}/orders.csv', discarded_folder)
    dbutils.notebook.exit('{"errorFlg": "true", "errorMsg": "Orderid is repeated"}')

ordersDf.createOrReplaceTempView("orders")


### 2nd Condition : move the file to discarded folder if order status doesn't match the ones stored in our sql database

#### setup database connection to lookup valid order status

In [0]:
dbServer = 'trendytechsqlserv'
dbPort = '1433'
dbName = 'free-sql-db-5673187'
dbUser = 'tt-sql-user'

connectionUrl = f'jdbc:sqlserver://{dbServer}.database.windows.net:{dbPort};database={dbName};user={dbUser};'

dbPassword =dbutils.secrets.get(scope = 'salesprojectscope', key='sql-password')

connectionProperties = {
'password': dbPassword,
'driver':'com.microsoft.sqlserver.jdbc.SQLServerDriver'
}

**only possible from interactive cluster or gen purp compute dont user serverless**

In [0]:
validStatusDf = spark.read.jdbc(url=connectionUrl, table='dbo.valid_order_status', properties= connectionProperties)

In [0]:
display(validStatusDf)

In [0]:
validStatusDf.createOrReplaceTempView("valid_status")

In [0]:
invalidRowsDf = spark.sql("select * from orders where order_status not in (select * from valid_status)")

In [0]:
display(invalidRowsDf)

In [0]:
if invalidRowsDf.count() > 0:
    errorFlag = True 

if errorFlag:
    dbutils.fs.mv(f'{landing_folder}/orders.csv', discarded_folder)
    dbutils.notebook.exit('{"errorFlg": "true", "errorMsg": "Invalid order status found"}')
else:
    dbutils.fs.mv(f'{landing_folder}/orders.csv', staging_folder)
    dbutils.notebook.exit('{"errorFlg": "false", "errorMsg": "All good"}')
